In [1]:
# DATA CLEANING — IEA Critical Minerals Dataset 2026 (South America)
#
# Goal: take 3 sheets of the IEA Critical Minerals Dataset 2026 from our GitHub repo, clean them, and combine them into one table for South America, saved as iea_data_clean.csv.
#
# Sheet — What it contains
# 2 Total supply for key minerals — how much of each mineral is mined and refined, by country, 2025–2040
# 3.1 Energy demand by tech — how much each clean-energy technology (EVs, solar, wind…) needs, 2025–2050, 3 scenarios
# 3.2 Energy demand by mineral — total clean-energy demand for each mineral, 2025–2050, 3 scenarios
#
# Why cleaning is needed: the IEA sheets are made to be read (titles, blank lines, tables side by side, headers inside the data, notes at the bottom), not to be analysed. We turn them into a long table: one row per number, with columns saying what the number is (mineral, stage, country, scenario, year).
#
# Units: all values are in kt (kilotonnes = thousand tonnes), except the "Top 3 share" rows, which are fractions (0.5 = 50 %).

In [2]:
# 1) Importing the libraries
#
# - pandas (pd) — to read the Excel file and work with tables.
# - numpy (np) — for small either/or choices (np.where) and empty values.
#
# We import them once at the top so every later cell can use them.

In [3]:
import pandas as pd
import numpy as np

In [4]:
# 2) Load the data
#
# What: read the three sheets straight from the raw GitHub link.
#
# Why: reading from the repo means everyone in the team runs the notebook on exactly the same file, with no local copies.
#
# How:
# - sheet_name= picks the sheet (the file has 12 sheets; we need only 3).
# - header=None — the sheets start with a title and empty rows, not with column names. So we read every cell as data and set the column names ourselves in the next steps.
#
# Output: the size (rows, columns) of each raw sheet.

In [5]:
# Load the raw data

url = "https://raw.githubusercontent.com/len-chou/TeamTokio---Week-3-Project-/refs/heads/main/IEA%20Critical%20Minerals%20Dataset%202026.xlsx"

df_supply_raw = pd.read_excel(url, sheet_name="2 Total supply for key minerals", header=None)
df_tech_raw = pd.read_excel(url, sheet_name="3.1 Energy demand by tech", header=None)
df_mineral_raw = pd.read_excel(url, sheet_name="3.2 Energy demand by mineral", header=None)

print("Supply sheet:", df_supply_raw.shape)
print("Demand by tech sheet:", df_tech_raw.shape)
print("Demand by mineral sheet:", df_mineral_raw.shape)

Supply sheet: (73, 11)
Demand by tech sheet: (97, 20)
Demand by mineral sheet: (46, 20)


In [6]:
# 3) Cleaning the supply sheet (2 Total supply for key minerals)
#
# First, look at the raw sheet (next cell) to understand its layout:
# - rows 0–2: empty rows and the title; row 3 holds the years 2025, 2030, 2035, 2040;
# - two tables side by side: mining in columns 0–4, refining in columns 6–10 (column 5 is an empty gap);
# - for each mineral a small block: a header like Copper - Mining, the country rows, Rest of world, Total, Top 3 share;
# - notes at the very bottom (text, not data).
#
# Each of the following sub-steps (3A–3E) fixes one of these problems.

In [7]:
df_supply_raw.head(16)

,0,1,2,3,4,5,6,7,8,9,10
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Total supply for key energy transition mineral...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025.000,2030.0,2035.000,2040.000,NaN,NaN,2025.000,2030.000,2035.000,2040.000
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Copper - Mining,NaN,NaN,NaN,NaN,NaN,Copper - Refining,NaN,NaN,NaN,NaN
6,Chile,5485.000,5369.0,4841.000,4165.000,NaN,China,13069.000,17121.000,17221.000,17221.000
7,Democratic Republic of Congo,3338.000,4212.0,3041.000,2696.000,NaN,Democratic Republic of Congo,2659.000,3278.000,2299.000,1979.000
8,Peru,2726.000,2712.0,2384.000,1859.000,NaN,Chile,1741.000,1846.000,1555.000,1359.000
9,China,1838.000,2040.0,1965.000,1852.000,NaN,Japan,1431.000,1624.000,1624.000,1624.000


In [8]:
# 3A. Put the two side-by-side tables under each other
#
# What: cut the sheet in two (left = mining, right = refining), give both halves the same column names (label, 2025, 2030, 2035, 2040) and stack them with pd.concat.
#
# Why: a table should have each kind of information in one column only. Side by side, the years and the countries appear twice.
#
# Also in this cell:
# - dropna(subset=["label"]) removes the empty rows (blank lines between blocks);
# - .str.strip() removes spaces before/after the text;
# - we remove the lines starting with "Notes" / "“Base case”" — they are the footnotes, not data.
#
# Output: the first rows of the stacked table. The header rows (Copper - Mining) are still there — we deal with them in 3B.

In [9]:
year_columns = ["2025", "2030", "2035", "2040"]

df_left = df_supply_raw.iloc[4:, 0:5].copy()      # mining
df_right = df_supply_raw.iloc[4:, 6:11].copy()    # refining

df_left.columns = ["label"] + year_columns
df_right.columns = ["label"] + year_columns

df_supply = pd.concat([df_left, df_right], ignore_index=True)

# remove the empty rows
df_supply = df_supply.dropna(subset=["label"])
df_supply["label"] = df_supply["label"].astype(str).str.strip()

# remove the notes at the bottom of the sheet
df_supply = df_supply[~df_supply["label"].str.startswith(("Notes", "“Base case”"))]

df_supply.head(12)

,label,2025,2030,2035,2040
1,Copper - Mining,NaN,NaN,NaN,NaN
2,Chile,5485.000,5369.0,4841.000,4165.000
3,Democratic Republic of Congo,3338.000,4212.0,3041.000,2696.000
4,Peru,2726.000,2712.0,2384.000,1859.000
5,China,1838.000,2040.0,1965.000,1852.000
6,Russia,1145.000,1244.0,1424.000,1343.000
7,United States,1101.000,1196.0,823.000,632.000
8,Rest of world,7594.000,7798.0,6255.000,4636.000
9,Total,23227.000,24571.0,20733.000,17183.000
10,Top 3 share,0.497,0.5,0.495,0.507


In [10]:
# 3B. Give every row its mineral and stage
#
# Problem: a row like Chile | 5485 doesn't say which mineral or stage it is — that information is only in the header row above it (Copper - Mining).
#
# What we do:
# 1. find the header rows: no number in 2025 and a " - " in the label;
# 2. keep the label only on those rows (where), then ffill() (forward fill) copies it down onto every row until the next header;
# 3. delete the header rows — they carry no numbers;
# 4. split Copper - Mining into two columns: mineral = Copper, stage = Mining.
#
# Output: number of rows per mineral. Most minerals have 18 rows (9 for mining + 9 for refining); nickel has 20 (one more producer listed), magnet rare earths 16 (one fewer).

In [11]:
is_header = df_supply["2025"].isna() & df_supply["label"].str.contains(" - ")

df_supply["mineral_stage"] = df_supply["label"].where(is_header)   # only the header rows keep a value
df_supply["mineral_stage"] = df_supply["mineral_stage"].ffill()    # copy it down to the rows below

df_supply = df_supply[~is_header].copy()                           # drop the header rows

df_supply[["mineral", "stage"]] = df_supply["mineral_stage"].str.split(" - ", n=1, expand=True)
df_supply = df_supply.drop(columns="mineral_stage")

df_supply["mineral"].value_counts()

mineral
Nickel                        20
Copper                        18
Cobalt                        18
Lithium                       18
Graphite                      18
Magnet rare earth elements    16
Name: count, dtype: int64

In [12]:
# 3C. What kind of row is it?
#
# Problem: the label column mixes real countries with summary rows. If we added everything up, the totals would be counted twice.
#
# What we do: a small function get_row_type gives each row a row_type:
# - country — a producing country;
# - rest_of_world — all the smaller producers together;
# - total — the world total (one block writes "Total clean technologies" instead of "Total" — it is the same kind of total, so any label starting with "Total" counts);
# - top3_share — the share of the 3 biggest producers (a fraction, not kt).
#
# We also create a country column: the country name, or "World" for totals and shares, because those numbers describe the whole world.
#
# Output: how many rows of each type.

In [13]:
def get_row_type(label):
    if label == "Rest of world":
        return "rest_of_world"
    if label.startswith("Total"):
        return "total"
    if label == "Top 3 share":
        return "top3_share"
    return "country"

df_supply["row_type"] = df_supply["label"].apply(get_row_type)

# the country column: the country name, or "World" for totals and shares
df_supply["country"] = df_supply["label"]
df_supply.loc[df_supply["row_type"].isin(["total", "top3_share"]), "country"] = "World"

df_supply["row_type"].value_counts()

row_type
country          72
rest_of_world    12
total            12
top3_share       12
Name: count, dtype: int64

In [14]:
# 3D. From wide to long
#
# Problem: each year is still its own column (2025, 2030…). That's "wide" format.
#
# What we do: melt turns the four year columns into two columns — year and value — so we get one row per number.
#
# Why: long format is easy to filter (year == 2030), group and plot, and it's the same shape as the demand tables, so we can stack them later.
#
# We also:
# - turn year into a whole number and value into a number;
# - add dataset = "supply" (which sheet the row comes from);
# - add scenario = "Base case" — the supply sheet has only one scenario.
#
# Output: 432 rows (108 rows × 4 years).

In [15]:
df_supply = df_supply.melt(
    id_vars=["mineral", "stage", "country", "row_type", "label"],
    value_vars=year_columns,
    var_name="year",
    value_name="value",
)

df_supply["year"] = df_supply["year"].astype(int)
df_supply["value"] = pd.to_numeric(df_supply["value"], errors="coerce")
df_supply["dataset"] = "supply"
df_supply["scenario"] = "Base case"          # supply only has one scenario

print("Shape:", df_supply.shape)
df_supply.head(30)

Shape: (432, 9)


,mineral,stage,country,row_type,label,year,value,dataset,scenario
0,Copper,Mining,Chile,country,Chile,2025,5485.000,supply,Base case
1,Copper,Mining,Democratic Republic of Congo,country,Democratic Republic of Congo,2025,3338.000,supply,Base case
2,Copper,Mining,Peru,country,Peru,2025,2726.000,supply,Base case
3,Copper,Mining,China,country,China,2025,1838.000,supply,Base case
4,Copper,Mining,Russia,country,Russia,2025,1145.000,supply,Base case
5,Copper,Mining,United States,country,United States,2025,1101.000,supply,Base case
6,Copper,Mining,Rest of world,rest_of_world,Rest of world,2025,7594.000,supply,Base case
7,Copper,Mining,World,total,Total,2025,23227.000,supply,Base case
8,Copper,Mining,World,top3_share,Top 3 share,2025,0.497,supply,Base case
9,Cobalt,Mining,Democratic Republic of Congo,country,Democratic Republic of Congo,2025,210.000,supply,Base case


In [16]:
# 3E. Check the supply numbers
#
# Why: to prove we read the right cells. If the sheet was parsed correctly, for every mineral, stage and year:
#
# countries + Rest of world = Total
#
# How to read the output: 48 groups are checked (12 mineral/stage blocks × 4 years). The biggest gap is 0.44 %. That's normal: the IEA rounds every number to 3 digits, so the parts never add up perfectly. No missing values.

In [17]:
parts = df_supply[df_supply["row_type"].isin(["country", "rest_of_world"])].groupby(["mineral", "stage", "year"])["value"].sum()
total = df_supply[df_supply["row_type"] == "total"].groupby(["mineral", "stage", "year"])["value"].sum()

gap = (parts - total).abs() / total
print("Mineral/stage/year groups checked:", len(gap))
print("Biggest gap between parts and total:", round(gap.max() * 100, 2), "%")
print("Missing values:", df_supply["value"].isna().sum())

Mineral/stage/year groups checked: 48
Biggest gap between parts and total: 0.44 %
Missing values: 0


In [18]:
# 3F. Keep only mining (no refining)
#
# Why: the project looks at where minerals are taken out of the ground. Mining is what happens in resource-rich areas and
# what can lead to conflicts over land. Refining (processing the ore into a usable material) often happens in other
# countries (for example China), so it is not part of our question and we remove it.
#
# Stages we keep: "Mining" and "Mining (natural)" (= natural graphite).
# Stages we remove: "Refining" and "Battery grade" (graphite processed for batteries, which is a refining step).
#
# We do this after the check in 3E, so the check still runs on the complete sheet.
#
# Output: the stages before and after, and the number of rows per mineral (now only mining).

In [19]:
mining_stages = ["Mining", "Mining (natural)"]

print("Stages before:", df_supply["stage"].unique())
df_supply = df_supply[df_supply["stage"].isin(mining_stages)].copy()
print("Stages after:", df_supply["stage"].unique())
print("Rows left:", len(df_supply))

df_supply["mineral"].value_counts()

Stages before: <StringArray>
['Mining', 'Mining (natural)', 'Refining', 'Battery grade']
Length: 4, dtype: str
Stages after: <StringArray>
['Mining', 'Mining (natural)']
Length: 2, dtype: str
Rows left: 216


mineral
Nickel                        40
Copper                        36
Cobalt                        36
Lithium                       36
Graphite                      36
Magnet rare earth elements    32
Name: count, dtype: int64

In [20]:
# 4) Cleaning the demand sheets (3.1 and 3.2)
#
# Look at the raw sheet (next cell). Both demand sheets have the same header:
# - one row with the scenario names, each written once above its block of columns (Current Policies in column 3, Stated Policies in column 9, High Demand in column 15);
# - one row with the years below it;
# - 2025 appears only once (column 1), before the scenarios — it is the real base year, the same for all scenarios;
# - empty columns (2, 8, 14) separate the blocks.
#
# The three scenarios: Current Policies = only policies already in force; Stated Policies = policies governments have announced; High Demand = faster clean-energy growth.

In [16]:
df_tech_raw.head(20)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Mineral demand for energy technologies - by mi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,Current Policies scenario,NaN,NaN,NaN,NaN,NaN,Stated Policies scenario,NaN,NaN,NaN,NaN,NaN,High Demand Scenario,NaN,NaN,NaN,NaN
4,NaN,2025.00000,NaN,2030,2035.0000,2040.0000,2045.00,2050.0000,NaN,2030,2035.0000,2040.0000,2045.00,2050.0000,NaN,2030,2035.000,2040.0000,2045.000,2050.000
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Chromium,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Wind,71.20000,NaN,82.5,72.1000,64.3000,74.20,85.8000,NaN,95.2,83.2000,70.5000,79.30,94.8000,NaN,133,122.000,108.0000,113.000,153.000
8,Other low emissions power generation,118.00000,NaN,254,271.0000,266.0000,271.00,246.0000,NaN,278,295.0000,293.0000,296.00,269.0000,NaN,363,353.000,358.0000,354.000,385.000
9,Total chromium,190.00000,NaN,336,343.0000,331.0000,345.00,332.0000,NaN,373,378.0000,363.0000,375.00,364.0000,NaN,496,474.000,466.0000,467.000,539.000


In [22]:
# 4A. Build the column names from the two header rows
#
# Problem: a value's meaning (scenario + year) is spread over two header rows, and the scenario name is written only once per block.
#
# What the function set_demand_columns does:
# 1. finds the scenario row (the one containing "Stated Policies scenario") — the years are in the row just below;
# 2. ffill() along that row copies each scenario name to the right over its whole block; the columns before the first scenario get "Base year (all scenarios)" (that's the 2025 column);
# 3. joins scenario and year into one name, e.g. Stated Policies scenario|2030;
# 4. keeps only the rows under the header, drops the empty separator columns and the empty rows.
#
# Why a function: sheets 3.1 and 3.2 have the same header, so we write the logic once and use it twice.
#
# Output: the first rows of sheet 3.1 with its new column names.

In [23]:
def set_demand_columns(df_raw):
    # the scenario row is the row that contains "Stated Policies scenario"; the years are in the row below
    scenario_row = df_raw[df_raw.eq("Stated Policies scenario").any(axis=1)].index[0]
    year_row = scenario_row + 1

    scenarios = df_raw.loc[scenario_row].ffill().fillna("Base year (all scenarios)")
    years = df_raw.loc[year_row]

    new_columns = []
    for scenario, year in zip(scenarios, years):
        if pd.isna(year):
            new_columns.append(None)                           # empty separator column
        else:
            new_columns.append(f"{scenario}|{int(year)}")
    new_columns[0] = "label"

    df = df_raw.iloc[year_row + 1:].copy()                     # keep only the rows under the header
    df.columns = new_columns
    df = df.loc[:, df.columns.notna()]                         # drop the separator columns
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(str).str.strip()
    return df

df_tech = set_demand_columns(df_tech_raw)
df_tech.head()

,label,Base year (all scenarios)|2025,Current Policies scenario|2030,Current Policies scenario|2035,Current Policies scenario|2040,Current Policies scenario|2045,Current Policies scenario|2050,Stated Policies scenario|2030,Stated Policies scenario|2035,Stated Policies scenario|2040,Stated Policies scenario|2045,Stated Policies scenario|2050,High Demand Scenario|2030,High Demand Scenario|2035,High Demand Scenario|2040,High Demand Scenario|2045,High Demand Scenario|2050
6,Chromium,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Wind,71.2,82.5,72.1,64.3,74.2,85.8,95.2,83.2,70.5,79.3,94.8,133,122.0,108.0,113.0,153.0
8,Other low emissions power generation,118.0,254,271.0,266.0,271.0,246.0,278,295.0,293.0,296.0,269.0,363,353.0,358.0,354.0,385.0
9,Total chromium,190.0,336,343.0,331.0,345.0,332.0,373,378.0,363.0,375.0,364.0,496,474.0,466.0,467.0,539.0
12,Copper,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# 4B. Demand by technology (3.1): give every row its mineral
#
# Same trick as 3B: in sheet 3.1 the mineral name (e.g. Chromium) is a header row with no numbers, and the technologies are listed below it.
#
# What we do:
# 1. header rows = rows where all values are empty;
# 2. ffill() copies the mineral name down, then the header rows are deleted;
# 3. rows starting with "Total" (e.g. Total copper) get row_type = total; the technology rows get row_type = component;
# 4. the stage column holds the technology name (Wind, Solar PV, Electric vehicles…), or "Total" for the total row.
#
# Output: number of rows per mineral (technologies + 1 total).

In [25]:
value_columns = [c for c in df_tech.columns if c != "label"]

is_header = df_tech[value_columns].isna().all(axis=1)
df_tech["mineral"] = df_tech["label"].where(is_header).ffill()
df_tech = df_tech[~is_header].copy()

df_tech["row_type"] = np.where(df_tech["label"].str.lower().str.startswith("total"), "total", "component")
df_tech["stage"] = np.where(df_tech["row_type"] == "total", "Total", df_tech["label"])   # the technology name

df_tech["mineral"].value_counts()

mineral
Copper                    8
Nickel                    7
Cobalt                    5
Manganese                 5
Molybdenum                4
Silicon                   4
Zinc                      4
Chromium                  3
Battery-grade graphite    3
Lithium                   3
Silver                    3
Neodymium                 3
PGMs                      2
Name: count, dtype: int64

In [26]:
# 4C. Demand by mineral (3.2)
#
# What: the same header function (set_demand_columns). Here each row is already one mineral, so no header rows to fill down.
#
# Why rename "Total": the last row, Total, is all minerals together. We rename it "All minerals" so nobody mistakes it for a mineral, and give it row_type = total. All other rows are component.
#
# stage = "All energy technologies" because this sheet doesn't split demand by technology.
#
# Output: 39 minerals, and the last rows of the table.

In [27]:
df_by_mineral = set_demand_columns(df_mineral_raw)

df_by_mineral["mineral"] = df_by_mineral["label"].replace({"Total": "All minerals"})
df_by_mineral["row_type"] = np.where(df_by_mineral["label"] == "Total", "total", "component")
df_by_mineral["stage"] = "All energy technologies"

print("Minerals:", df_by_mineral["mineral"].nunique())
df_by_mineral.tail()

Minerals: 39


,label,Base year (all scenarios)|2025,Current Policies scenario|2030,Current Policies scenario|2035,Current Policies scenario|2040,Current Policies scenario|2045,Current Policies scenario|2050,Stated Policies scenario|2030,Stated Policies scenario|2035,Stated Policies scenario|2040,Stated Policies scenario|2045,Stated Policies scenario|2050,High Demand Scenario|2030,High Demand Scenario|2035,High Demand Scenario|2040,High Demand Scenario|2045,High Demand Scenario|2050,mineral,row_type,stage
41,Terbium,0.46300,0.713,0.81200,0.74400,0.80000,0.89100,0.804,0.94200,0.89900,1.01000,1.19000,1.12,1.2800,1.3500,1.4900,1.84000,Terbium,component,All energy technologies
42,Yttrium,0.00308,0.0128,0.01380,0.01180,0.00973,0.00785,0.0132,0.01510,0.01340,0.01120,0.00935,0.0232,0.0208,0.0145,0.0132,0.00668,Yttrium,component,All energy technologies
43,Lanthanum,0.00394,0.00514,0.00411,0.00326,0.00314,0.00432,0.00514,0.00411,0.00326,0.00314,0.00432,0.0357,0.0256,0.0258,0.0363,0.05150,Lanthanum,component,All energy technologies
44,Total rare earth elements,20.90000,33.1,38.00000,35.80000,37.40000,39.90000,36.6,44.10000,44.40000,49.50000,56.10000,48.4,59.7000,64.5000,70.0000,80.30000,Total rare earth elements,component,All energy technologies
45,Total,13391.00000,18422,21194.00000,22297.00000,22990.00000,23142.00000,19711,23918.00000,25979.00000,27078.00000,27885.00000,23128,29185.0000,33014.0000,33863.0000,36510.00000,All minerals,total,All energy technologies


In [28]:
# 4D. Same thing as 3D: the function demand_to_long (maybe optional)
# 1. melts all the scenario|year columns into one column;
# 2. splits Stated Policies scenario|2030 back into scenario and year;
# 3. makes year a whole number and value a number;
# 4. sets country = "World" — the IEA gives demand only for the whole world, not per country;
# 5. adds the dataset name.
#
# Output: 864 rows for 3.1 and 624 for 3.2 (each row of the sheet × 16 values: 1 base year + 5 years × 3 scenarios), and the count per scenario.

In [29]:
def demand_to_long(df, dataset_name):
    value_columns = [c for c in df.columns if "|" in c]
    df_long = df.melt(
        id_vars=["mineral", "stage", "row_type", "label"],
        value_vars=value_columns,
        var_name="scenario_year",
        value_name="value",
    )
    df_long[["scenario", "year"]] = df_long["scenario_year"].str.split("|", expand=True)
    df_long["year"] = df_long["year"].astype(int)
    df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
    df_long["country"] = "World"
    df_long["dataset"] = dataset_name
    return df_long.drop(columns="scenario_year")

df_tech = demand_to_long(df_tech, "demand_by_technology")
df_by_mineral = demand_to_long(df_by_mineral, "demand_by_mineral")

print("Demand by technology:", df_tech.shape)
print("Demand by mineral:", df_by_mineral.shape)
df_tech["scenario"].value_counts()

Demand by technology: (864, 9)
Demand by mineral: (624, 9)


scenario
Current Policies scenario    270
Stated Policies scenario     270
High Demand Scenario         270
Base year (all scenarios)     54
Name: count, dtype: int64

In [30]:
# 4E. Check the demand numbers
#
# Why: to prove the scenario/year columns were matched correctly and that the two sheets agree.
#
# 1. In 3.1, the technologies should add up to the "Total" row of each mineral.
# 2. The totals in 3.1 should equal the values in 3.2 — it's the same demand, shown two ways.
#    Sheet 3.1 calls one mineral "PGMs" and sheet 3.2 "PGMs (other than iridum)"; the numbers are identical, so we match the names for this check.
# 3. No missing values, no negative values (demand can't be negative).
#
# Result: the technologies are at most 2.7 % off the total for a few minerals (copper, manganese, cobalt) — the IEA rounds every number and doesn't list every small use. We keep the IEA totals as published. The 3.1 totals match 3.2 exactly (208 of 208, 0 % gap).

In [31]:
keys = ["mineral", "scenario", "year"]

parts = df_tech[df_tech["row_type"] == "component"].groupby(keys)["value"].sum()
total = df_tech[df_tech["row_type"] == "total"].groupby(keys)["value"].sum()
gap = (parts - total).abs() / total
print("1) Biggest gap technologies vs total:", round(gap.max() * 100, 2), "%")

total_3_2 = df_by_mineral.assign(mineral=df_by_mineral["mineral"].replace({"PGMs (other than iridum)": "PGMs"}))
total_3_2 = total_3_2.groupby(keys)["value"].sum()
compare = pd.concat([total, total_3_2], axis=1, keys=["sheet_3_1", "sheet_3_2"], join="inner")
print("2) Minerals matched:", len(compare), "of", len(total),
      "| biggest gap:", round(((compare["sheet_3_1"] - compare["sheet_3_2"]).abs() / compare["sheet_3_2"]).max() * 100, 2), "%")

print("Missing values:", df_tech["value"].isna().sum() + df_by_mineral["value"].isna().sum())
print("Negative values:", (df_tech["value"] < 0).sum() + (df_by_mineral["value"] < 0).sum())

1) Biggest gap technologies vs total: 2.7 %
2) Minerals matched: 208 of 208 | biggest gap: 0.0 %
Missing values: 0
Negative values: 0


In [32]:
# 5) Combine the three tables
#
# What: pd.concat stacks the supply table and the two demand tables into one table, df_clean. They have the same columns, and dataset says which sheet each row came from.
#
# Why: to compare supply with demand, both must be in the same table, with the same column and mineral names.
#
# Text cleaning in this cell:
# - .str.strip() — remove stray spaces;
# - fix the IEA typo "iridum" → "iridium";
# - the total labels mix capitals ("Total copper", "Total Lithium") → .str.capitalize() gives them all the same style;
# - add mineral_group, a common name linking supply and demand where the IEA uses different names:
#
# mineral_group — name in supply — name(s) in demand
# Graphite — Graphite — Battery-grade graphite
# Magnet rare earths — Magnet rare earth elements — Neodymium, Praseodymium, Dysprosium, Terbium (the rare earths used in magnets)
# PGMs (other than iridium) — — — PGMs (3.1), PGMs (other than iridium) (3.2)
#
# - label is no longer needed (the names are now in country / stage), so we keep it only for totals, as total_label;
# - unit = "kt", or "share" for the Top 3 share rows.
#
# Output: 1,920 rows, 0 duplicates, and the number of rows from each sheet.

In [33]:
df_clean = pd.concat([df_supply, df_tech, df_by_mineral], ignore_index=True)

# text cleaning
for col in ["mineral", "stage", "country", "label", "scenario"]:
    df_clean[col] = df_clean[col].str.strip()

df_clean["mineral"] = df_clean["mineral"].str.replace("iridum", "iridium")
df_clean["label"] = df_clean["label"].str.capitalize()

# same mineral names for supply and demand
mineral_groups = {
    "Battery-grade graphite": "Graphite",
    "Magnet rare earth elements": "Magnet rare earths",
    "Neodymium": "Magnet rare earths",
    "Praseodymium": "Magnet rare earths",
    "Dysprosium": "Magnet rare earths",
    "Terbium": "Magnet rare earths",
    "PGMs": "PGMs (other than iridium)",
}
df_clean["mineral_group"] = df_clean["mineral"].replace(mineral_groups)

# we don't need "label" for countries and technologies any more (they are in country / stage) — keep it only for totals
df_clean["total_label"] = df_clean["label"].where(df_clean["row_type"] == "total")
df_clean = df_clean.drop(columns="label")

df_clean["unit"] = np.where(df_clean["row_type"] == "top3_share", "share", "kt")

print("Shape:", df_clean.shape)
print("Duplicated rows:", df_clean.duplicated(["dataset", "mineral", "stage", "country", "row_type", "scenario", "year"]).sum())
df_clean["dataset"].value_counts()

Shape: (1704, 11)
Duplicated rows: 0


dataset
demand_by_technology    864
demand_by_mineral       624
supply                  216
Name: count, dtype: int64

In [34]:
# 6) Keep South America — all countries, all minerals, all years
#
# Why: the project is about South America. But the IEA supply sheet only names the biggest producers of each mineral: in South America only Argentina, Brazil, Chile and Peru appear. All other countries (Colombia, Bolivia, Venezuela…) are hidden inside "Rest of world". To still have every South American country for every mineral and year, we build a complete grid.
#
# What the cell does:
# 1. south_america — the list of the 12 South American countries;
# 2. df_sup — only the supply rows;
# 3. combos — every mineral × stage × year that exists in the supply sheet (24 combinations, mining only);
# 4. df_grid — every country × every combination (how="cross"): 12 × 24 = 288 rows;
# 5. merge the IEA numbers into the grid (how="left"): countries the IEA lists get their number;
# 6. fillna(0) — countries/minerals with no IEA figure get 0. ⚠️ Here 0 means "the IEA does not report this country separately", not necessarily zero production (e.g. Bolivia's lithium is inside "Rest of world");
# 7. df_demand keeps demand rows whose country is not "World";
# 8. pd.concat combines South American supply and that demand into the new df_clean.
#
# Output: the size of the table, the list of countries, and the number of supply rows per country.

In [35]:
# 6) Keep South America — all countries, all minerals, all years

# 6) Keep South America — all countries, all minerals, all years

south_america = [
    "Argentina",
    "Bolivia",
    "Brazil",
    "Chile",
    "Colombia",
    "Ecuador",
    "Guyana",
    "Paraguay",
    "Peru",
    "Suriname",
    "Uruguay",
    "Venezuela"
]

# Look only at supply data
df_sup = df_clean[df_clean["dataset"] == "supply"].copy()

# Create the combinations that exist in the IEA supply data
combos = (
    df_sup[df_sup["row_type"] == "country"]
    [
        [
            "dataset",
            "mineral_group",
            "mineral",
            "stage",
            "scenario",
            "year",
            "unit"
        ]
    ]
    .drop_duplicates()
)

# Create all South American countries × all IEA supply combinations
df_grid = pd.DataFrame({
    "country": south_america
}).merge(
    combos,
    how="cross"
)

df_grid["row_type"] = "country"

# Add the actual IEA values
df_sa_supply = df_grid.merge(
    df_sup[df_sup["country"].isin(south_america)][
        ["country", "mineral", "stage", "year", "value"]
    ],
    on=["country", "mineral", "stage", "year"],
    how="left"
)

# If the IEA has no value, use 0
df_sa_supply["value"] = df_sa_supply["value"].fillna(0)

# Keep demand data, but remove World
df_demand = df_clean[
    (df_clean["dataset"] != "supply")
    & (df_clean["country"] != "World")
].copy()

# Combine South American supply + demand
df_clean = pd.concat(
    [df_sa_supply, df_demand],
    ignore_index=True
)

# Check result
print("Shape:", df_clean.shape)

print("\nCountries in final dataset:")
print(sorted(df_clean["country"].unique()))

print("\nSupply observations by country:")
print(
    df_clean[df_clean["dataset"] == "supply"]
    .groupby("country")["value"]
    .agg(["size", "count"])
    .rename(columns={
        "size": "rows",
        "count": "rows_with_value"
    })
)

Shape: (288, 11)

Countries in final dataset:
['Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela']

Supply observations by country:
           rows  rows_with_value
country                         
Argentina    24               24
Bolivia      24               24
Brazil       24               24
Chile        24               24
Colombia     24               24
Ecuador      24               24
Guyana       24               24
Paraguay     24               24
Peru         24               24
Suriname     24               24
Uruguay      24               24
Venezuela    24               24


In [36]:
# 7) Final tidy-up
#
# What: put the columns in a logical order (what it is → where → when → how much), sort the rows, and check the result with info().
#
# How to read info(): the number of rows, each column's type, and how many values are not empty in each column.

In [ ]:
df_clean = df_clean[["dataset", "mineral_group", "mineral", "stage", "country", "row_type", "total_label",
                     "scenario", "year", "value", "unit", "reported_by_iea"]]
df_clean = df_clean.sort_values(["dataset", "mineral_group", "mineral", "stage", "country", "scenario", "year"]).reset_index(drop=True)

print("--- Data Info ---")
df_clean.info()

In [ ]:
df_clean.head(10)

In [ ]:
# 8) Quick look: South America's share of world supply
#
# What: for each country and mineral, divide the country's supply by the World total for the same mineral, stage and year.
#
# Why: a quick check that the table makes sense, and an example of how to use it.
#
# How to read it: values are fractions — 0.236 = 23.6 %. Chile mines about a quarter of the world's copper, Peru about a tenth, and Argentina's share of lithium mining roughly doubles by 2040.

In [ ]:
df_sa = df_clean[(df_clean["dataset"] == "supply") & (df_clean["country"] != "World") & df_clean["reported_by_iea"]]
df_world = df_clean[(df_clean["dataset"] == "supply") & (df_clean["row_type"] == "total")]

df_share = df_sa.merge(df_world[["mineral", "stage", "year", "value"]], on=["mineral", "stage", "year"], suffixes=("", "_world"))
df_share["share_of_world"] = (df_share["value"] / df_share["value_world"]).round(3)

df_share.pivot_table(index=["country", "mineral", "stage"], columns="year", values="share_of_world")

In [ ]:
# 9) Save the cleaned data to CSV
#
# What: write df_clean to iea_data_clean.csv. index=False leaves out pandas' row numbers, which aren't data.

In [ ]:
df_clean.to_csv("iea_data_clean.csv", index=False)
print("Saved iea_data_clean.csv with", len(df_clean), "rows.")

In [ ]:
# Things to remember when using iea_data_clean.csv
# - The IEA reports supply only for Argentina, Brazil, Chile and Peru. In step 6 the other South American countries get 0, which means not reported separately by the IEA — not necessarily no production.
# - Supply covers 2025–2040 (one scenario); demand covers 2025–2050 (3 scenarios) — compare them only up to 2040.
# - Demand is only for clean-energy technologies, not every use of a mineral.
# - Filter on row_type before adding numbers, so totals aren't counted twice. The real total is in column "world"